# Train the PBMC SLE model (Colab or Kaggle)

Runs the project pipeline from this repo end to end, then (optionally) uploads the trained model so the
Hugging Face demo can use it.

1. **Colab:** Runtime → Change runtime type → **T4 GPU**. **Kaggle:** Session options → GPU T4 and Internet On.
2. **Run all.** It asks for a GitHub token (the repo is private) and, only if `UPLOAD = True`, a Hugging Face write token.

Steps: data (download + QC + HVGs) → represent (scVI, CellTypist, UMAP) → train (baseline vs attention-MIL, donor-split CV) → insights.

In [ ]:
import socket
try:
    socket.create_connection(("pypi.org", 443), timeout=5).close(); print("Internet OK")
except OSError:
    raise RuntimeError("No internet in this notebook session. Kaggle: Session options > Internet On "
                       "(phone-verified account). Or use Colab, where internet is always on.")

In [ ]:
# ---- settings ----
MAX_CELLS_PER_DONOR = 1000   # lower (e.g. 500) if the session runs out of RAM
UPLOAD = False               # True: push artifacts/ to the Hugging Face model repo the demo reads
MODEL_REPO = "AdamSion74/pbmc-sle-model"

In [ ]:
import os, sys, getpass, subprocess
os.chdir("/content" if os.path.exists("/content") else "/kaggle/working")
if not os.path.exists("immune-research"):
    token = getpass.getpass("GitHub token with read access to adamsi/immune-research: ")
    subprocess.run(["git", "clone", "--depth", "1", f"https://{token}@github.com/adamsi/immune-research.git"], check=True)
os.chdir("immune-research")
# Colab has old pandas/numpy already loaded, so restart once after installing (the clone is kept).
if not os.path.exists(".deps_installed"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    "projects/pbmc_sle_classification/requirements.txt"], check=True)
    open(".deps_installed", "w").close()
    print("Installed. Restarting the runtime now. When it reconnects, click Run all again.")
    os.kill(os.getpid(), 9)
print("Packages ready")

In [ ]:
import yaml
cfg = yaml.safe_load(open("projects/pbmc_sle_classification/configs/default.yaml"))
cfg["data"]["max_cells_per_donor"] = MAX_CELLS_PER_DONOR
yaml.safe_dump(cfg, open("projects/pbmc_sle_classification/configs/notebook.yaml", "w"))
CFG = "projects/pbmc_sle_classification/configs/notebook.yaml"

In [ ]:
!python -m projects.pbmc_sle_classification.data --config {CFG}

In [ ]:
!python -m projects.pbmc_sle_classification.represent --config {CFG}

In [ ]:
!python -m projects.pbmc_sle_classification.train --config {CFG}

In [ ]:
!python -m projects.pbmc_sle_classification.insights --config {CFG}

In [ ]:
import json
from IPython.display import Image, display
import pandas as pd
R = "projects/pbmc_sle_classification/results"
print(json.dumps(json.load(open(f"{R}/metrics.json")), indent=2))
display(pd.read_csv(f"{R}/attention_by_celltype.csv", index_col=0).head(10))
display(Image(f"{R}/umap.png"))

In [ ]:
if UPLOAD:
    from huggingface_hub import HfApi
    api = HfApi(token=getpass.getpass("Hugging Face write token: "))
    api.create_repo(MODEL_REPO, repo_type="model", exist_ok=True)
    api.upload_folder(repo_id=MODEL_REPO, folder_path="projects/pbmc_sle_classification/artifacts",
                      repo_type="model", ignore_patterns=["scvi/*"])
    print(f"Uploaded to https://huggingface.co/{MODEL_REPO}")
print("Commit projects/pbmc_sle_classification/results/metrics.json to show the scores in the demo.")